<a href="https://colab.research.google.com/github/venkateshsunkula/pre-recall-radar/blob/main/pre_recall_radar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pre-Recall Radar
An AI agent that detects emerging product safety issues from public CPSC complaint data, before official recalls.

## Step 1: Download CPSC data
Full public database from SaferProducts.gov (complaints + recalls).

In [ ]:
!wget -O SPDB.zip https://www.saferproducts.gov/SPDB.zip
!unzip -o SPDB.zip
!ls

--2026-10-08 01:47:12--  https://www.saferproducts.gov/SPDB.zip
Resolving www.saferproducts.gov (www.saferproducts.gov)... 23.213.217.144, 2600:1417:76:4a0::983, 2600:1417:76:4a2::983
Connecting to www.saferproducts.gov (www.saferproducts.gov)|23.213.217.144|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 39661443 (38M) [application/x-zip-compressed]
Saving to: ‘SPDB.zip’

SPDB.zip            100%[===================>]  37.82M  20.4MB/s    in 1.8s    

2026-10-08 01:47:15 (20.4 MB/s) - ‘SPDB.zip’ saved [39661443/39661443]

Archive:  SPDB.zip
  inflating: Disclaimer.txt          
  inflating: IncidentReports.csv     
  inflating: Recalls.csv             
Disclaimer.txt	IncidentReports.csv  Recalls.csv  sample_data  SPDB.zip


## Step 2: Load complaints
The CSV has a disclaimer line above the header, so we find the real header row first.

In [ ]:
import pandas as pd

with open("IncidentReports.csv", encoding="latin-1") as f:
    for i, line in enumerate(f):
        if "Report No." in line:
            header_row = i
            break

print("Real header is on line", header_row)

df = pd.read_csv("IncidentReports.csv", skiprows=header_row,
                 low_memory=False, encoding="latin-1")
df.columns = df.columns.str.strip()

print(df.shape)
print([c for c in df.columns if "Category" in c])

Real header is on line 1
(70000, 40)
['Category of Submitter', 'Product Category', 'Product Sub Category']


## Step 3: Focus on baby products

In [ ]:
print(df["Product Category"].value_counts().head(20))

Product Category
Kitchen                                 25204
Home Maintenance and Structures         11727
Furniture, Furnishings & Decorations     6016
Electronics                              5441
Baby                                     4672
Sports and Recreation                    4485
Toys & Children                          3607
Yard & Garden                            3447
Clothing and Accessories                 1925
Personal Care                            1199
Containers & Packaging                   1130
Fuel, Lighters and Fireworks              574
Hobby                                     429
Products at Public Facilities              96
Drywall                                    48
Name: count, dtype: int64


In [ ]:
baby = df[
    df["Product Category"].str.contains("nursery|infant|baby", case=False, na=False)
    | df["Product Sub Category"].str.contains("nursery|infant|baby|crib|stroller|car seat", case=False, na=False)
].copy()

baby["Report Date"] = pd.to_datetime(baby["Report Date"], errors="coerce")

print(len(baby), "baby product complaints")
print(baby["Product Type"].value_counts().head(15))
baby[["Report Date", "Brand", "Product Type", "Incident Description"]].head()

5324 baby product complaints
Product Type
Diapers (1512)                                                         942
Bassinets or Cradles (1537)                                            597
Baby Strollers (1522)                                                  595
Nonportable Cribs (1543)                                               448
Pacifiers or Teething Rings (1525)                                     430
High Chairs (1555)                                                     254
Baby Exercisers (1520)                                                 210
Infant & Toddler Play Ctrs, Excl Jumpers,bouncers&exercisers (1550)    175
Baby Walkers or Jumpers (1508)                                         169
Baby Gates or Barriers (1506)                                          163
Night-lights (1533)                                                    151
Playpens (1513)                                                        149
Baby Bouncer Seats (Excl. Jumpers) (1558)                 

,Report Date,Brand,Product Type,Incident Description
31,2026-09-18,Millie moon,Diapers (1512),Millie moon diapers were used on my daughter a...
32,2026-09-18,Millie Moon,Diapers (1512),We have been using Millie Moon diapers since o...
33,2026-09-19,Millie Moon,Diapers (1512),My 7 week old daughter had a Millie moon diape...
34,2026-09-19,Millie Moon,Diapers (1512),My daughter received what looks like chemical ...
36,2026-09-18,Millie Moon,Diapers (1512),Opened a box of Millie moon diapers and was hi...


## Step 4: Spike detector
Flags products with 3+ complaints in the last 90 days AND 3x their usual monthly rate.

In [ ]:
# 1. Clean brand names: lowercase, letters only ("Millie Moon" -> "milliemoon")
baby["brand_key"] = (baby["Brand"].astype(str).str.lower()
                     .str.replace(r"[^a-z]", "", regex=True))

# 2. Define time windows
today = baby["Report Date"].max()             # newest complaint = "today"
recent_start = today - pd.Timedelta(days=90)     # last 3 months
before_start = recent_start - pd.Timedelta(days=365)  # the year before that

recent = baby[(baby["Report Date"] > recent_start) &
              (baby["Report Date"] <= today)]
before = baby[(baby["Report Date"] > before_start) &
              (baby["Report Date"] <= recent_start)]

# 3. Count complaints per product in each window
keys = ["brand_key", "Product Type"]
spikes = pd.concat([
    recent.groupby(keys).size().rename("recent_90d"),
    before.groupby(keys).size().rename("prior_365d"),
], axis=1).fillna(0)

# 4. Flag: at least 3 recent complaints AND 3x more than usual
spikes["recent_per_month"] = spikes["recent_90d"] / 3
spikes["usual_per_month"] = spikes["prior_365d"] / 12
spikes["flag"] = ((spikes["recent_90d"] >= 3) &
                  (spikes["recent_per_month"] >= 3 * spikes["usual_per_month"]))

flagged = spikes[spikes["flag"]].sort_values("recent_90d", ascending=False)
print(len(flagged), "products flagged")
flagged.head(15)


4 products flagged


,,recent_90d,prior_365d,recent_per_month,usual_per_month,flag
brand_key,Product Type,,,,,
milliemoon,Diapers (1512),292.0,207.0,97.333333,17.250000,True
milliemoonluxurydiapers,Diapers (1512),10.0,10.0,3.333333,0.833333,True
millemoon,Diapers (1512),6.0,3.0,2.000000,0.250000,True
pampers,Diapers (1512),3.0,3.0,1.000000,0.250000,True


## Step 5: Claude as the judge
Claude reads complaints and decides if they describe the same hazard. API key is stored in Colab Secrets, never in code.

In [ ]:
!pip install -q anthropic
import anthropic, json
from google.colab import userdata
client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 4.9 MB/s eta 0:00:00


In [ ]:
def judge(brand_key, product_type):
    rows = recent[(recent["brand_key"] == brand_key) &
                  (recent["Product Type"] == product_type)]
    texts = rows["Incident Description"].dropna().astype(str).str[:500].head(15).tolist()
    complaints = "\n\n".join(f"- {t}" for t in texts)

    prompt = f"""You are a product-safety analyst. Below are recent consumer complaints about one product.

Brand as typed by consumers: {brand_key}
Product type: {product_type}
Number of complaints: {len(texts)}

Complaints:
{complaints}

Rules:
- canonical_brand must be a cleaned-up version of the brand as typed above, company/brand name ONLY (drop product-line words like "Luxury", "Diapers", "Size 4"). Never use a different brand mentioned in the complaint text.
- If same_hazard is false, severity must be "low".
- If there are fewer than 5 complaints, confidence must be 0.5 or lower.

Reply with ONLY a JSON object, no other text:
{{"canonical_brand": "clean brand name",
 "same_hazard": true or false,
 "hazard": "short description of the main hazard, or null",
 "severity": "low" or "medium" or "high",
 "confidence": number between 0 and 1,
 "summary": "2 factual sentences. Say 'reports describe', not 'the product causes'."}}"""

    resp = client.messages.create(model="claude-haiku-4-5-20251001", max_tokens=500,
                                  messages=[{"role": "user", "content": prompt}])
    text = resp.content[0].text.strip()
    text = text.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(text)

results = []
for (brand_key, product_type), row in flagged.iterrows():
    verdict = judge(brand_key, product_type)
    verdict.update({"brand_key": brand_key, "recent_90d": int(row["recent_90d"])})
    results.append(verdict)
    print(verdict["canonical_brand"], "|", verdict["hazard"], "|", verdict["severity"])

judged = pd.DataFrame(results)
judged

Millie Moon | Chemical burns and severe skin reactions in diaper area | high
Millie Moon | Chemical burn-like severe rash and blistering on infant genital and diaper-contact areas | high
Millie Moon | Severe diaper rash and chemical burn-like reactions in genital and buttock areas | high
Pampers | Multiple distinct hazards reported: severe allergic skin reactions with blistering, and foreign object (metal screw) contamination | low


,canonical_brand,same_hazard,hazard,severity,confidence,summary,brand_key,recent_90d
0,Millie Moon,True,Chemical burns and severe skin reactions in di...,high,0.95,"Reports describe chemical burns, blisters, and...",milliemoon,292
1,Millie Moon,True,Chemical burn-like severe rash and blistering ...,high,0.95,Reports describe severe chemical burn-like ras...,milliemoonluxurydiapers,10
2,Millie Moon,True,Severe diaper rash and chemical burn-like reac...,high,0.75,"Reports describe bright red, painful rashes an...",millemoon,6
3,Pampers,False,Multiple distinct hazards reported: severe all...,low,0.50,Reports describe two distinct safety concerns ...,pampers,3


### Evaluation: checking Claude's output
Claude invented a brand name that appears in none of the complaints, so it was a hallucination. Fixed by adding rules to the prompt.

In [ ]:
rows = recent[recent["brand_key"] == "millemoon"]
print(rows["Brand"].unique())
for t in rows["Incident Description"].astype(str).head(6):
    print("-", t[:300], "\n")

['Mille Moon' 'Mille moon ']
- Bright red rash on babys butt and genital areas, that was painful to her after a few days of use. 

- While using Millie Moon diapers, I observed an increase in my babys frequent, bright red rashes. During a routine check-up, my pediatrician also noticed the issue and expressed concern. Diaper changes became increasingly challenging, as my baby became fussy and vocal during the process of changing h 

- Ive been using Millie moon diapers for my baby since I got cut open seven layers of skin and have never had any issues. My daughter is now 9month as old and just like any other regular day my daughter needed more diapers. I so happened to notice a change in the Millie moon diapers with the pattern g 

- My son has developed a very bad diaper rash after three days of using Millie moon diapers from target. 

- Ive been using Millie Moon since my son was born he got a bad rash on July 31st of 2026 I didnt think anything of it I dont really be on social media 

In [ ]:
print(rows["Incident Description"].astype(str).str.contains("mama bear", case=False).sum())

0


## Step 6: Final alerts
Keep only real patterns and merge brand spellings.

In [ ]:
alerts = (judged[judged["same_hazard"] == True]
          .groupby("canonical_brand")
          .agg(complaints_90d=("recent_90d", "sum"),
               hazard=("hazard", "first"),
               severity=("severity", "first"),
               summary=("summary", "first"))
          .sort_values("complaints_90d", ascending=False))
alerts["level"] = alerts["complaints_90d"].apply(lambda n: "🚨 Alert" if n >= 10 else "👀 Watch")
alerts

,complaints_90d,hazard,severity,summary,level
canonical_brand,,,,,
Millie Moon,308,Chemical burns and severe skin reactions in di...,high,"Reports describe chemical burns, blisters, and...",🚨 Alert


## Step 7: Recall check

In [ ]:
def recall_status(brand):
    hits = [line.strip()[:200]
            for line in open("Recalls.csv", encoding="latin-1")
            if brand.lower() in line.lower()]
    status = "⚠️ Brand has past recall(s): verify it's the same product" if hits else "⚠️ EARLY SIGNAL - no recall found"
    return status, hits

for brand in alerts.index:
    status, hits = recall_status(brand)
    alerts.loc[brand, "status"] = status
    print(brand, "→", status)
    for h in hits[:3]:
        print("    ", h)

alerts

Millie Moon → ⚠️ EARLY SIGNAL - no recall found


,complaints_90d,hazard,severity,summary,level,status
canonical_brand,,,,,,
Millie Moon,308,Chemical burns and severe skin reactions in di...,high,"Reports describe chemical burns, blisters, and...",🚨 Alert,⚠️ EARLY SIGNAL - no recall found


## Step 8: Backtest
Replays history week by week, using only data that was public at the time.

In [ ]:
baby["Publication Date"] = pd.to_datetime(baby["Publication Date"], errors="coerce")

def is_flagged(day, brand="milliemoon", date_col="Publication Date"):
    d = baby[date_col]
    rs = day - pd.Timedelta(days=90)
    bs = rs - pd.Timedelta(days=365)
    mine = baby["brand_key"] == brand
    recent_n = (mine & (d > rs) & (d <= day)).sum()
    before_n = (mine & (d > bs) & (d <= rs)).sum()
    flag = recent_n >= 3 and (recent_n / 3) >= 3 * (before_n / 12)
    return flag, recent_n, before_n

for col in ["Report Date", "Publication Date"]:
    for day in pd.date_range("2026-01-01", "2026-09-30", freq="W"):
        flag, r, b = is_flagged(day, date_col=col)
        if flag:
            print(f"{col}: first flagged {day.date()} | recent 90d: {r} | prior year: {b}")
            break

Report Date: first flagged 2026-03-29 | recent 90d: 3 | prior year: 1
Publication Date: first flagged 2026-04-26 | recent 90d: 3 | prior year: 1


### Lead time vs. false alarms
Watch level (3+) = early but noisy. Alert level (10+) = later but precise.

In [ ]:
def first_flag(min_n, brand="milliemoon", col="Publication Date"):
    for day in pd.date_range("2026-01-01", "2026-09-30", freq="W"):
        d = baby[col]
        rs = day - pd.Timedelta(days=90); bs = rs - pd.Timedelta(days=365)
        mine = baby["brand_key"] == brand
        r = (mine & (d > rs) & (d <= day)).sum()
        b = (mine & (d > bs) & (d <= rs)).sum()
        if r >= min_n and r/3 >= 3 * b/12:
            return day.date(), r

def false_alarms(min_n, col="Publication Date"):
    d = baby[col]
    flagged = set()
    for day in pd.date_range("2026-01-01", "2026-09-30", freq="W"):
        rs = day - pd.Timedelta(days=90); bs = rs - pd.Timedelta(days=365)
        r = baby[(d > rs) & (d <= day)].groupby("brand_key").size()
        b = baby[(d > bs) & (d <= rs)].groupby("brand_key").size()
        both = pd.concat([r.rename("r"), b.rename("b")], axis=1).fillna(0)
        hit = both[(both.r >= min_n) & (both.r/3 >= 3 * both.b/12)].index
        flagged.update(hit)
    return len(flagged)

for level, n in [("Watch", 3), ("Alert", 10)]:
    print(f"{level} (>= {n}): Millie Moon first flagged {first_flag(n)} | "
          f"total brands flagged Jan-Sep: {false_alarms(n)}")

Watch (>= 3): Millie Moon first flagged (datetime.date(2026, 4, 26), np.int64(3)) | total brands flagged Jan-Sep: 14
Alert (>= 10): Millie Moon first flagged (datetime.date(2026, 5, 24), np.int64(14)) | total brands flagged Jan-Sep: 3


### Which brands triggered the Alert level?

In [ ]:
d = baby["Publication Date"]
alert_brands = set()
for day in pd.date_range("2026-01-01", "2026-09-30", freq="W"):
    rs = day - pd.Timedelta(days=90); bs = rs - pd.Timedelta(days=365)
    r = baby[(d > rs) & (d <= day)].groupby("brand_key").size()
    b = baby[(d > bs) & (d <= rs)].groupby("brand_key").size()
    both = pd.concat([r.rename("r"), b.rename("b")], axis=1).fillna(0)
    alert_brands.update(both[(both.r >= 10) & (both.r/3 >= 3 * both.b/12)].index)
print(alert_brands)


{'ciuseianx', 'milliemoonluxurydiapers', 'milliemoon'}


### Investigating the second alert
Infant walkers: CPSC issued a safety warning on Aug 6, 2026. The agent alerted on Jul 5, 32 days earlier.

In [ ]:
print("Watch:", first_flag(3, brand="ciuseianx"))
print("Alert:", first_flag(10, brand="ciuseianx"))

Watch: (datetime.date(2026, 6, 7), np.int64(4))
Alert: (datetime.date(2026, 7, 5), np.int64(10))


In [ ]:
c = baby[baby["brand_key"] == "ciuseianx"]
print(len(c), "complaints |", c["Product Type"].unique())
for t in c["Incident Description"].astype(str).head(5):
    print("-", t[:250], "\n")
print(recall_status("ciuseianx"))

11 complaints | ['Baby Exercisers (1520)' 'Baby Walkers or Jumpers (1508)']
- We are a distributor of maternity and baby products operating on Amazon US. We only engage in product distribution and are not a manufacturer. Back in [REDACTED], we sold infant walkers covered under Recall [REDACTED]. From [REDACTED] to the present, 

- I bought a baby walker for my sisters child on Amazon. Id come across some nice baby walkers on [REDACTED] before, and my nephew was old enough to use one. So I placed an order on Amazon. But it ended up being a terrible mistake. What happened is som 

- I bought this product last month, planning to take my daughter on vacation with it this month. Id hoped to watch her use the walker while we were away. I was initially drawn to its appearance, but what happened over the past month is something I'll n 

- My grandson is turning six months old soon. I picked out a lovely gift for his baby shower and specially bought a baby walker for him. I hoped it would help 

## v2 - Step 1: Check every flag against official recalls and warnings

In [ ]:
!wget -q -O warnings.csv https://www.cpsc.gov/s3fs-public/recall-data/product_safety_warning_listing.csv
!wget -q -O recalls_cpsc.csv https://www.cpsc.gov/s3fs-public/recall-data/recalls_recall_listing.csv
!ls -lh warnings.csv recalls_cpsc.csv

-rw-r--r-- 1 root root  18M Sep 25 20:29 recalls_cpsc.csv
-rw-r--r-- 1 root root 1.1M Oct  7 13:27 warnings.csv


In [ ]:
import re

def flagged_brands(min_n, col="Publication Date"):
    d = baby[col]
    first_seen = {}
    for day in pd.date_range("2026-01-01", "2026-09-30", freq="W"):
        rs = day - pd.Timedelta(days=90); bs = rs - pd.Timedelta(days=365)
        r = baby[(d > rs) & (d <= day)].groupby("brand_key").size()
        b = baby[(d > bs) & (d <= rs)].groupby("brand_key").size()
        both = pd.concat([r.rename("r"), b.rename("b")], axis=1).fillna(0)
        for brand in both[(both.r >= min_n) & (both.r/3 >= 3 * both.b/12)].index:
            first_seen.setdefault(brand, day.date())
    return first_seen

official = []
for fname, kind in [("warnings.csv", "WARNING"), ("recalls_cpsc.csv", "RECALL")]:
    for line in open(fname, encoding="latin-1"):
        official.append((kind, re.sub(r"[^a-z]", "", line.lower()), line.strip()[:150]))

for brand, first in sorted(flagged_brands(3).items(), key=lambda x: x[1]):
    hits = [(k, t) for k, norm, t in official
            if len(brand) >= 4 and brand != "nan" and brand in norm]
    result = f"{hits[0][0]}: {hits[0][1]}" if hits else "no official action found"
    print(f"{first}  {brand:26} → {result}")

2026-01-04  cradlewise                 → no official action found
2026-02-15  wolfcode                   → WARNING: 26-391,"April 9, 2026","CPSC Warns Consumers to Stop Using Wolfcode Infant Walkers Immediately Due to Risk of Serious Injury or Death from Fall; Viola
2026-02-22  smartsteps                 → no official action found
2026-04-26  milliemoon                 → no official action found
2026-05-03  fisherprice                → RECALL: 25-216,,"April 10, 2025",,"Fisher-Price Recalls Tissue Box Toy Sold with SnugaPuppy Activity Centers Due to Choking Hazard","Tissue Box Toy sold with 
2026-05-03  joolbaby                   → RECALL: 24-145,,"March 07, 2024",,"Jool Baby Recalls Nova Baby Infant Swings Due to Suffocation Hazard; Violation of the Federal Safety Regulations","Jool Bab
2026-05-10  nuby                       → RECALL: 25-160,,"March 06, 2025",,"Luv n' care Recalls Nuby Stroller Fans Due to Laceration Injury Hazard","Nuby Stroller Fans","This recall involves Nuby Str
2

## v2 - Step 2: The investigator agent
Claude gets tools and decides on its own what to check, in what order, and when to stop.

In [ ]:
def norm(s):
    return re.sub(r"[^a-z]", "", str(s).lower())

def get_complaints(brand):
    rows = baby[baby["brand_key"].str.contains(norm(brand), na=False)]
    if rows.empty:
        return "No complaints found for this brand."
    monthly = rows.groupby(rows["Publication Date"].dt.to_period("M")).size().tail(12)
    texts = (rows.sort_values("Publication Date", ascending=False)["Incident Description"]
             .dropna().astype(str).str[:300].head(8))
    return (f"Total complaints: {len(rows)}\n"
            f"Product types: {rows['Product Type'].value_counts().head(3).to_dict()}\n"
            f"Complaints per month: {{{', '.join(f'{p}: {n}' for p, n in monthly.items())}}}\n"
            "Most recent complaints:\n" + "\n".join(f"- {t}" for t in texts))

def check_official_actions(brand):
    b = norm(brand)
    hits = [f"{kind}: {text}" for kind, n, text in official if len(b) >= 4 and b in n]
    return "\n".join(hits[:6]) if hits else "No official CPSC recalls or warnings found for this brand."

brand_input = {"type": "object",
               "properties": {"brand": {"type": "string", "description": "Brand name, e.g. 'Flyboss'"}},
               "required": ["brand"]}

TOOLS = [
    {"name": "get_complaints",
     "description": "Get CPSC consumer complaints for a brand: total count, product types, complaints per month, and recent complaint texts.",
     "input_schema": brand_input},
    {"name": "check_official_actions",
     "description": "Search official CPSC recalls and product safety warnings for a brand. Returns titles with dates. Matches may be old or about a DIFFERENT product from the same brand, so judge relevance yourself.",
     "input_schema": brand_input},
    {"type": "web_search_20250305", "name": "web_search", "max_uses": 3},
]
def count_mentions(brand, keyword):
    rows = baby[baby["brand_key"].str.contains(norm(brand), na=False)]
    hits = rows["Incident Description"].astype(str).str.contains(keyword, case=False).sum()
    return f"{hits} of {len(rows)} complaints mention '{keyword}'."

TOOLS.append({
    "name": "count_mentions",
    "description": "Count how many of a brand's complaints mention a keyword (e.g. 'collapse', 'strangle', 'bolt'). Use this to count hazards across ALL complaints, not just the samples.",
    "input_schema": {"type": "object",
                     "properties": {"brand": {"type": "string"},
                                    "keyword": {"type": "string"}},
                     "required": ["brand", "keyword"]}})

In [ ]:
SYSTEM = """You are a product-safety investigator. Our spike detector flagged a brand because consumer complaints suddenly increased.

Investigate with your tools:
1. Read the complaints. Do they describe the same hazard?
2. Check official CPSC actions. Only count an action if it is about the SAME product and hazard. Note its date.
3. Search the web for news or other reports (max 3 searches).

Then write a short report:
HAZARD: one line
EVIDENCE: 3 bullets with numbers
OFFICIAL STATUS: action + date, or none
NEWS: one line, with source names
CONFIDENCE: low / medium / high, and why

Rules:
- Say "reports describe", never "the product causes". Never invent facts. If a tool returns nothing, say so.
- Only state numbers that came directly from a tool. You only see a SAMPLE of complaint texts; use the TOTAL from get_complaints, and use count_mentions to count hazards across all complaints.
- When count_mentions gives a number, report THAT number (e.g. "34 of 41 complaints mention collapse"), not the sample count.
- Label each piece of evidence by WHERE the number came from:
  [CPSC complaints] = only numbers returned by get_complaints or count_mentions
  [official action] = numbers from CPSC warnings/recalls (even if found via web search)
  [news] / [social media - unverified]

At the very end of your report, add ONE line exactly in this format (JSON, true/false only):
FACTS: {"one_main_hazard": true, "official_action_same_product": false, "credible_news": false}
- one_main_hazard: most complaints describe the SAME specific hazard
- official_action_same_product: a CPSC recall/warning about this exact product and hazard
- credible_news: real news outlets (not social media, not law-firm websites)
Do NOT write a verdict. Our code decides it from these facts."""

TOOL_FUNCTIONS = {"get_complaints": get_complaints,
                  "check_official_actions": check_official_actions,
                  "count_mentions": count_mentions}

def investigate(brand, max_steps=10):
    messages = [{"role": "user", "content": f"Investigate this flagged brand: {brand}"}]
    for step in range(1, max_steps + 1):
        resp = client.messages.create(model="claude-haiku-4-5-20251001", max_tokens=2000,
                                      system=SYSTEM, tools=TOOLS, messages=messages)
        messages.append({"role": "assistant", "content": resp.content})

        for block in resp.content:
            if block.type == "tool_use":
                print(f"🔧 Step {step}: {block.name}({block.input})")
            elif block.type == "server_tool_use":
                print(f"🌐 Step {step}: web_search({block.input})")

        if resp.stop_reason == "tool_use":
            results = []
            for block in resp.content:
                if block.type == "tool_use":
                    output = TOOL_FUNCTIONS[block.name](**block.input)
                    print(f"     ↳ {str(output)[:150]}")
                    results.append({"type": "tool_result", "tool_use_id": block.id, "content": output})
            messages.append({"role": "user", "content": results})
        elif resp.stop_reason == "pause_turn":
            continue
        else:
            return "".join(b.text for b in resp.content if b.type == "text")
    return "Stopped: reached the step limit."

## v2 - Step 3: Final verdicts
The agent extracts facts; Python applies the verdict rules, the same way every time.

In [ ]:
def complaint_count(brand):
    return int(baby["brand_key"].str.contains(norm(brand), na=False).sum())

def decide(brand, report):
    m = re.search(r"FACTS:\s*(\{.*?\})", report, re.S)
    if not m:
        return "UNCLEAR (agent gave no FACTS line)"
    f = json.loads(m.group(1))
    n = complaint_count(brand)
    if not f.get("one_main_hazard"):
        return "UNCLEAR: several different hazards"
    if f.get("official_action_same_product") or f.get("credible_news") or n >= 10:
        return "LIKELY REAL ISSUE"
    return "UNCLEAR: one hazard, but weak evidence so far"

for brand in ["Flyboss", "Cradlewise", "Smartsteps"]:
    print(f"\n{'='*60}\n{brand}\n{'='*60}")
    report = investigate(brand)
    print(report)
    print(f"\n>>> FINAL VERDICT (rules applied in code): {decide(brand, report)}")


Flyboss
🔧 Step 1: get_complaints({'brand': 'Flyboss'})
🔧 Step 1: check_official_actions({'brand': 'Flyboss'})
     ↳ Total complaints: 8
Product types: {'Baby Bouncer Seats (Excl. Jumpers) (1558)': 8}
Complaints per month: {2025-12: 1, 2026-01: 1, 2026-04: 1, 2026-05
     ↳ WARNING: 26-757,"September 10, 2026","CPSC Warns Consumers to Stop Using Flyboss Infant Bouncers Immediately Due to Risk of Collapse and Impact Injury
🔧 Step 2: count_mentions({'brand': 'Flyboss', 'keyword': 'collapse'})
🌐 Step 2: web_search({'query': 'Flyboss baby bouncer CPSC recall 2026'})
🌐 Step 2: web_search({'query': 'Flyboss infant bouncer collapse warning'})
     ↳ 7 of 8 complaints mention 'collapse'.
Perfect. I have comprehensive data. Let me compile the report.

---

## INVESTIGATION REPORT: FLYBOSS

**HAZARD:** Flyboss 3-in-1 baby bouncers collapse unexpectedly during use, causing infants to fall and strike their heads on underlying surfaces.

**EVIDENCE:**

• 7 of 8 CPSC complaints mention collapse [CP

## v2 - Step 4: Watchlist + Telegram alerts
Pick brands you own; the agent checks them and sends the result to your phone.

In [ ]:
import requests
TG_TOKEN = userdata.get("TELEGRAM_TOKEN")
TG_CHAT = userdata.get("TELEGRAM_CHAT_ID")

def send_telegram(text):
    r = requests.post(f"https://api.telegram.org/bot{TG_TOKEN}/sendMessage",
                      data={"chat_id": TG_CHAT, "text": text[:4000]})
    return r.json().get("ok")

print(send_telegram("Pre-Recall Radar is connected ✅"))

True


In [ ]:
def latest_official_date(brand):
    hits = check_official_actions(brand)
    dates = re.findall(r'"([A-Z][a-z]+ \d{1,2}, \d{4})"', hits)
    return max(pd.to_datetime(dates)) if dates else None

In [ ]:
WATCHLIST = ["Millie Moon", "Flyboss", "Cradlewise", "Graco"]

def spike_status(brand, col="Publication Date"):
    d = baby[col]
    day = d.max()
    rs = day - pd.Timedelta(days=90); bs = rs - pd.Timedelta(days=365)
    mine = baby["brand_key"].str.contains(norm(brand), na=False)
    r = int((mine & (d > rs) & (d <= day)).sum())
    b = int((mine & (d > bs) & (d <= rs)).sum())
    spiking = r >= 3 and r / 3 >= 3 * b / 12
    if spiking and r >= 10:
        level = "🚨 Alert"
    elif spiking:
        level = "👀 Watch"
    else:
        level = "✅ Quiet"
    return level, r, b

def run_watchlist(deep=True):
    lines = [f"Pre-Recall Radar: watchlist check (data up to {baby['Publication Date'].max().date()})"]
    for brand in WATCHLIST:
        level, r, b = spike_status(brand)
        lines.append(f"\n{brand}: {level}\n  Last 90 days: {r} complaints (prior year: {b})")
        last = latest_official_date(brand)
        if last is not None and last >= baby["Publication Date"].max() - pd.Timedelta(days=365):
            level = "⛔ OFFICIAL WARNING/RECALL - stop using, check cpsc.gov"
            lines[-1] = f"\n{brand}: {level}\n  Last 90 days: {r} complaints (prior year: {b})"
        elif last is not None:
            lines.append(f"  Past official action: {last.date()} (may be a different product)")
        if deep and level != "✅ Quiet":
            report = investigate(brand)
            lines.append(f"  Verdict: {decide(brand, report)}")
    msg = "\n".join(lines)
    print(msg)
    print("Sent to Telegram:", send_telegram(msg))

run_watchlist()

🔧 Step 1: get_complaints({'brand': 'Millie Moon'})
🔧 Step 1: check_official_actions({'brand': 'Millie Moon'})
     ↳ Total complaints: 536
Product types: {'Diapers (1512)': 536}
Complaints per month: {2023-10: 1, 2023-11: 1, 2025-09: 1, 2025-11: 1, 2026-02: 1, 2026-0
     ↳ No official CPSC recalls or warnings found for this brand.
🔧 Step 2: count_mentions({'brand': 'Millie Moon', 'keyword': 'rash'})
🔧 Step 2: count_mentions({'brand': 'Millie Moon', 'keyword': 'blister'})
🔧 Step 2: count_mentions({'brand': 'Millie Moon', 'keyword': 'burn'})
     ↳ 386 of 536 complaints mention 'rash'.
     ↳ 98 of 536 complaints mention 'blister'.
     ↳ 294 of 536 complaints mention 'burn'.
🌐 Step 3: web_search({'query': 'Millie Moon diapers rash skin reactions 2026'})
🌐 Step 3: web_search({'query': 'Millie Moon diapers blister formula change'})
🌐 Step 3: web_search({'query': 'Millie Moon Target diapers recall complaints'})
🔧 Step 1: get_complaints({'brand': 'Flyboss'})
🔧 Step 1: check_official_action

## Weekly data refresh
saferproducts.gov blocks GitHub's servers, so Colab downloads the data and saves it to the repo.

In [1]:
import base64, gzip, requests
import pandas as pd
from google.colab import userdata

!wget -q -O SPDB.zip https://www.saferproducts.gov/SPDB.zip && unzip -o -q SPDB.zip
!wget -q -O warnings.csv https://www.cpsc.gov/s3fs-public/recall-data/product_safety_warning_listing.csv
!wget -q -O recalls_cpsc.csv https://www.cpsc.gov/s3fs-public/recall-data/recalls_recall_listing.csv

# Keep only baby-product complaints and the columns radar.py uses
with open("IncidentReports.csv", encoding="latin-1") as f:
    header_row = next(i for i, line in enumerate(f) if "Report No." in line)
df = pd.read_csv("IncidentReports.csv", skiprows=header_row, low_memory=False, encoding="latin-1")
df.columns = df.columns.str.strip()
baby_df = df[
    df["Product Category"].str.contains("nursery|infant|baby", case=False, na=False)
    | df["Product Sub Category"].str.contains("nursery|infant|baby|crib|stroller|car seat", case=False, na=False)
]
cols = ["Report Date", "Publication Date", "Brand", "Product Type", "Incident Description"]
print(len(baby_df), "baby complaints")

# Save each file into the repo's data/ folder
REPO = "venkateshsunkula/pre-recall-radar"
GH = {"Authorization": f"Bearer {userdata.get('GH_TOKEN')}",
      "Accept": "application/vnd.github+json"}

def push(path, data):
    url = f"https://api.github.com/repos/{REPO}/contents/{path}"
    body = {"message": f"Weekly data refresh: {path}",
            "content": base64.b64encode(gzip.compress(data)).decode()}
    old = requests.get(url, headers=GH)
    if old.status_code == 200:
        body["sha"] = old.json()["sha"]
    r = requests.put(url, headers=GH, json=body)
    print(path, "✅" if r.status_code in (200, 201) else f"❌ {r.status_code} {r.text[:200]}")

push("data/baby.csv.gz", baby_df[cols].to_csv(index=False).encode())
push("data/warnings.csv.gz", open("warnings.csv", "rb").read())
push("data/recalls_cpsc.csv.gz", open("recalls_cpsc.csv", "rb").read())

5362 baby complaints
data/baby.csv.gz ✅
data/warnings.csv.gz ✅
data/recalls_cpsc.csv.gz ✅
